# 02_20 Kaggle ResidualUNet3D PHE-only fair baseline

This notebook trains a 3D residual U-Net-style baseline for PHE segmentation on Kaggle T4. The implementation uses MONAI `SegResNet`, a maintained 3D residual encoder-decoder segmentation network, as the ResidualUNet3D model family.

Fairness lock for comparison with `02_12`, `18b`, and related PHE baselines:

- Same dataset: PHE-SICH CT only.
- Same target: binary manual PHE mask (`background=0`, `PHE=1`).
- Same locked split: train 99, val 10, test 11.
- No ICH teacher, no pseudo ICH label, no prior channel, no external ICH supervision.
- Main changed factor: ResidualUNet3D architecture.

MONAI is used for the model, transforms, loss, sliding-window inference, and validation metrics. Torch/CUDA from Kaggle are kept intact.


In [1]:
from pathlib import Path
import os
import sys
import re
import json
import time
import math
import random
import shutil
import subprocess
import gc

import numpy as np
import pandas as pd

IS_KAGGLE = Path('/kaggle/input').exists()
KAGGLE_INPUT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working') if IS_KAGGLE else Path.cwd().resolve()
LOCAL_ROOT = Path.cwd().resolve()

OUTPUT_ROOT = WORK_ROOT / 'outputs_02_20_kaggle_residualunet3d_phe_only_fair_baseline'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
PRED_DIR = OUTPUT_ROOT / 'predictions_best_model'
METRIC_DIR = OUTPUT_ROOT / 'metrics'
CACHE_DIR = OUTPUT_ROOT / 'monai_cache'
for p in [OUTPUT_ROOT, CHECKPOINT_DIR, PRED_DIR, METRIC_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Optional manual override if auto-detect picks the wrong Kaggle dataset folder.
# Example:
# USER_PHE_ROOT = Path('/kaggle/input/phe-sich-ct-ids/PHE-SICH-CT-IDS/SubdatasetA_NIFIT/NIFIT')
USER_PHE_ROOT = None

# Locked 02_12 split: train 99, val 10, test 11.
TRAIN_SCAN_IDS = {
    '0001', '0004', '0005', '0006', '0008', '0009', '0011', '0012', '0015', '0016',
    '0017', '0018', '0021', '0022', '0023', '0024', '0025', '0026', '0028', '0030',
    '0031', '0032', '0034', '0035', '0036', '0037', '0038', '0039', '0040', '0041',
    '0042', '0043', '0044', '0045', '0046', '0048', '0049', '0052', '0053', '0055',
    '0056', '0057', '0058', '0059', '0064', '0065', '0066', '0067', '0068', '0069',
    '0070', '0071', '0073', '0074', '0075', '0076', '0077', '0081', '0083', '0086',
    '0087', '0088', '0090', '0092', '0093', '0097', '0098', '0099', '0100', '0101',
    '0102', '0103', '0104', '0106', '0108', '0112', '0118', '0124', '0125', '0126',
    '0129', '0131', '0132', '0133', '0134', '0135', '0136', '0139', '0140', '0141',
    '0142', '0144', '0146', '0161', '0162', '0163', '0164', '0165', '0166'
}
VAL_SCAN_IDS = {'0013', '0014', '0060', '0078', '0080', '0115', '0119', '0130', '0138', '0160'}
TEST_SCAN_IDS = {'0002', '0029', '0033', '0051', '0061', '0084', '0095', '0096', '0113', '0116', '0167'}
EXPECTED_SCAN_IDS = TRAIN_SCAN_IDS | VAL_SCAN_IDS | TEST_SCAN_IDS

SEED = 20260220
MODEL_NAME = 'monai_segresnet_residualunet3d_phe_only'
MAX_EPOCHS = 250
VAL_INTERVAL = 5
BATCH_SIZE = 1
NUM_SAMPLES_PER_CASE = 2
NUM_WORKERS = 2 if IS_KAGGLE else 0
CACHE_RATE_TRAIN = 0.10 if IS_KAGGLE else 0.0
CACHE_RATE_VAL_TEST = 0.0

# T4-safe defaults. ROI dims are compatible with the residual encoder-decoder downsampling path.
ROI_SIZE = (96, 96, 32)
SW_BATCH_SIZE = 4
SLIDING_OVERLAP = 0.50
BASE_CHANNELS = 24
DROPOUT = 0.05
AMP = True
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
GRAD_CLIP_NORM = 12.0

RUN_DEP_SETUP = True
CLONE_MONAI_REPO = False
INSTALL_MONAI = True
MONAI_PACKAGE_SPEC = 'monai>=1.5.0'
MONAI_REPO_URL = 'https://github.com/Project-MONAI/MONAI.git'
MONAI_CLONE_DIR = WORK_ROOT / 'MONAI'

RUN_TRAIN = True
RESUME_TRAINING = True
RUN_PREDICT_AND_EVALUATE = True

print('IS_KAGGLE       :', IS_KAGGLE)
print('WORK_ROOT       :', WORK_ROOT)
print('OUTPUT_ROOT     :', OUTPUT_ROOT)
print('Split sizes     :', len(TRAIN_SCAN_IDS), len(VAL_SCAN_IDS), len(TEST_SCAN_IDS))
print('ROI_SIZE        :', ROI_SIZE)
print('MAX_EPOCHS      :', MAX_EPOCHS)
print('BASE_CHANNELS   :', BASE_CHANNELS)
print('DROPOUT         :', DROPOUT)
print('MODEL_NAME      :', MODEL_NAME)


IS_KAGGLE       : True
WORK_ROOT       : /kaggle/working
OUTPUT_ROOT     : /kaggle/working/outputs_02_20_kaggle_residualunet3d_phe_only_fair_baseline
Split sizes     : 99 10 11
ROI_SIZE        : (96, 96, 32)
MAX_EPOCHS      : 250
BASE_CHANNELS   : 24
DROPOUT         : 0.05
MODEL_NAME      : monai_segresnet_residualunet3d_phe_only


## 0. Kaggle setup

This cell avoids the old U-Mamba dependency trap. It keeps Kaggle's existing Torch/CUDA stack, installs a current MONAI package, and optionally clones the official MONAI repository for reproducibility/provenance.


In [2]:
def run_cmd(cmd, cwd=None, check=True):
    print('>>>', ' '.join(map(str, cmd)))
    t0 = time.time()
    result = subprocess.run(cmd, cwd=str(cwd) if cwd is not None else None, text=True)
    print(f'Done in {(time.time() - t0) / 60:.1f} min, returncode={result.returncode}')
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd)
    return result


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
        if torch.cuda.is_available():
            torch.backends.cuda.matmul.allow_tf32 = True
    except Exception as exc:
        print('Torch seed setup skipped:', repr(exc))


if RUN_DEP_SETUP:
    if IS_KAGGLE and CLONE_MONAI_REPO:
        if not MONAI_CLONE_DIR.exists():
            clone_result = run_cmd(['git', 'clone', '--depth', '1', MONAI_REPO_URL, str(MONAI_CLONE_DIR)], check=False)
            if clone_result.returncode != 0:
                print('MONAI clone failed or internet is disabled. Continuing with pip package if available.')
        else:
            print('MONAI repo already exists:', MONAI_CLONE_DIR)

    if INSTALL_MONAI:
        # --no-deps keeps Kaggle Torch/CUDA intact. This avoids the NCCL/Torch mismatch seen in the old U-Mamba attempt.
        pip_result = run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '-U', '--no-deps', MONAI_PACKAGE_SPEC, 'nibabel', 'tqdm'], check=False)
        if pip_result.returncode != 0:
            print('MONAI pip install failed. Continuing only if a compatible MONAI package is already available.')

seed_everything(SEED)

import torch
import nibabel as nib
import scipy
import monai

print('Python :', sys.version.replace('\n', ' '))
print('Torch  :', torch.__version__)
print('CUDA   :', torch.version.cuda)
print('MONAI  :', monai.__version__)
print('Nibabel:', nib.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


>>> /usr/bin/python3 -m pip install -q -U --no-deps monai>=1.5.0 nibabel tqdm
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 6.0 MB/s eta 0:00:00
Done in 0.1 min, returncode=0


2026-06-21 13:13:56.539422: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782047636.717316      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782047636.775588      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782047637.221272      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782047637.221311      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782047637.221314      23 computation_placer.cc:177] computation placer alr

Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch  : 2.10.0+cu128
CUDA   : 12.8
MONAI  : 1.5.2
Nibabel: 5.4.2
GPU available: True
GPU name: Tesla T4
GPU memory GB: 14.56


## 1. Locate PHE-SICH and build locked split manifest

The notebook accepts the Kaggle dataset folder automatically. If it cannot find `set/` and `label/`, set `USER_PHE_ROOT` in the first cell.


In [3]:
def case_id_from_path(path):
    name = Path(path).name
    name = re.sub(r'\.nii(\.gz)?$', '', name)
    m = re.search(r'(\d{4})', name)
    return m.group(1) if m else name


def has_phe_layout(root):
    root = Path(root)
    return (root / 'set').is_dir() and (root / 'label').is_dir()


def find_phe_root():
    candidates = []
    if USER_PHE_ROOT is not None:
        candidates.append(Path(USER_PHE_ROOT))

    base_roots = [
        LOCAL_ROOT / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
        WORK_ROOT / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
    ]
    candidates.extend(base_roots)

    if IS_KAGGLE and KAGGLE_INPUT.exists():
        for dataset_root in sorted([p for p in KAGGLE_INPUT.iterdir() if p.is_dir()]):
            candidates.extend([
                dataset_root / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
                dataset_root / 'SubdatasetA_NIFIT' / 'NIFIT',
                dataset_root / 'NIFIT',
                dataset_root,
            ])

    seen = set()
    for cand in candidates:
        cand = cand.resolve() if cand.exists() else cand
        if str(cand) in seen:
            continue
        seen.add(str(cand))
        if has_phe_layout(cand):
            return cand

    if IS_KAGGLE and KAGGLE_INPUT.exists():
        for cand in KAGGLE_INPUT.rglob('NIFIT'):
            if has_phe_layout(cand):
                return cand

    raise FileNotFoundError('Could not find PHE-SICH NIFIT root with set/ and label/. Set USER_PHE_ROOT manually.')


def build_index(folder):
    files = sorted(list(Path(folder).glob('*.nii')) + list(Path(folder).glob('*.nii.gz')))
    return {case_id_from_path(p): p for p in files}


def split_name(case_id):
    if case_id in TEST_SCAN_IDS:
        return 'test'
    if case_id in VAL_SCAN_IDS:
        return 'val'
    if case_id in TRAIN_SCAN_IDS:
        return 'train'
    raise ValueError(f'Case {case_id} is not in the locked split.')

PHE_ROOT = find_phe_root()
IMAGE_DIR = PHE_ROOT / 'set'
MASK_DIR = PHE_ROOT / 'label'
image_by_id = build_index(IMAGE_DIR)
mask_by_id = build_index(MASK_DIR)

missing_images = sorted(EXPECTED_SCAN_IDS - set(image_by_id))
missing_masks = sorted(EXPECTED_SCAN_IDS - set(mask_by_id))
if missing_images:
    raise FileNotFoundError(f'Missing expected CT images: {missing_images[:20]}')
if missing_masks:
    raise FileNotFoundError(f'Missing expected PHE masks: {missing_masks[:20]}')

records = []
for case_id in sorted(EXPECTED_SCAN_IDS):
    records.append({
        'case_id': case_id,
        'split': split_name(case_id),
        'image': str(image_by_id[case_id]),
        'label': str(mask_by_id[case_id]),
    })

split_df = pd.DataFrame(records)
split_csv = OUTPUT_ROOT / '02_20_residualunet3d_locked_split_manifest.csv'
split_df.to_csv(split_csv, index=False)

print('PHE_ROOT:', PHE_ROOT)
print('Images  :', len(image_by_id))
print('Masks   :', len(mask_by_id))
print('Manifest:', split_csv)
print(split_df.groupby('split').size())
display(split_df.head())


PHE_ROOT: /kaggle/input/datasets/onanhv/phe-sich-ct-ids/PHE-SICH-CT-IDS/SubdatasetA_NIFIT/NIFIT
Images  : 120
Masks   : 120
Manifest: /kaggle/working/outputs_02_20_kaggle_residualunet3d_phe_only_fair_baseline/02_20_residualunet3d_locked_split_manifest.csv
split
test     11
train    99
val      10
dtype: int64


,case_id,split,image,label
0,0001,train,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...
1,0002,test,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...
2,0004,train,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...
3,0005,train,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...
4,0006,train,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...


## 2. Quick data sanity check

This is not required for training, but it confirms shape/spacing and whether masks are non-empty.


In [4]:
def load_nifti_info(path):
    img = nib.load(str(path))
    arr = np.asanyarray(img.dataobj)
    return {
        'shape': tuple(int(x) for x in arr.shape),
        'spacing': tuple(float(x) for x in img.header.get_zooms()[:3]),
        'min': float(np.nanmin(arr)),
        'max': float(np.nanmax(arr)),
        'nonzero': int(np.count_nonzero(arr)),
    }

sample_rows = []
for row in split_df.groupby('split', sort=False).head(2).itertuples(index=False):
    image_info = load_nifti_info(row.image)
    label_info = load_nifti_info(row.label)
    sample_rows.append({
        'case_id': row.case_id,
        'split': row.split,
        'image_shape': image_info['shape'],
        'spacing': image_info['spacing'],
        'ct_min': image_info['min'],
        'ct_max': image_info['max'],
        'phe_voxels': label_info['nonzero'],
    })

qc_df = pd.DataFrame(sample_rows)
qc_csv = OUTPUT_ROOT / '02_20_residualunet3d_data_qc_sample.csv'
qc_df.to_csv(qc_csv, index=False)
print('QC sample:', qc_csv)
display(qc_df)


QC sample: /kaggle/working/outputs_02_20_kaggle_residualunet3d_phe_only_fair_baseline/02_20_residualunet3d_data_qc_sample.csv


,case_id,split,image_shape,spacing,ct_min,ct_max,phe_voxels
0,0001,train,"(512, 512, 32)","(0.5078125, 0.5078125, 5.0)",-1024.0,1874.0,7962
1,0002,test,"(512, 512, 32)","(0.48828125, 0.48828125, 5.0)",-1024.0,3071.0,1635
2,0004,train,"(512, 512, 32)","(0.4296875, 0.4296875, 5.0)",-1024.0,3071.0,3441
3,0013,val,"(512, 512, 32)","(0.50390625, 0.50390625, 5.0)",-1024.0,1854.0,7467
4,0014,val,"(512, 512, 32)","(0.4296875, 0.4296875, 5.0)",-1024.0,1826.0,5070
5,0029,test,"(512, 512, 32)","(0.4296875, 0.4296875, 5.0)",-1024.0,2201.0,4222


## 3. MONAI datasets and transforms

We train from random 3D patches but validate/test with full-volume sliding-window inference. No extra label source is used.


In [5]:
import torch
from torch.utils.data import DataLoader
from monai.data import CacheDataset, Dataset, list_data_collate
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    ScaleIntensityRanged,
    Lambdad,
    SpatialPadd,
    RandCropByPosNegLabeld,
    RandFlipd,
    RandRotate90d,
    RandScaleIntensityd,
    RandShiftIntensityd,
    EnsureTyped,
)

train_files = split_df[split_df['split'] == 'train'][['image', 'label', 'case_id']].to_dict('records')
val_files = split_df[split_df['split'] == 'val'][['image', 'label', 'case_id']].to_dict('records')
test_files = split_df[split_df['split'] == 'test'][['image', 'label', 'case_id']].to_dict('records')

base_transforms = [
    LoadImaged(keys=['image', 'label'], image_only=True),
    EnsureChannelFirstd(keys=['image', 'label']),
    ScaleIntensityRanged(keys=['image'], a_min=-100.0, a_max=200.0, b_min=0.0, b_max=1.0, clip=True),
    Lambdad(keys=['label'], func=lambda x: (x > 0).astype(np.int64)),
    SpatialPadd(keys=['image', 'label'], spatial_size=ROI_SIZE),
]

train_transforms = Compose(base_transforms + [
    RandCropByPosNegLabeld(
        keys=['image', 'label'],
        label_key='label',
        spatial_size=ROI_SIZE,
        pos=2,
        neg=1,
        num_samples=NUM_SAMPLES_PER_CASE,
        image_key='image',
        image_threshold=0.01,
    ),
    RandFlipd(keys=['image', 'label'], prob=0.50, spatial_axis=0),
    RandFlipd(keys=['image', 'label'], prob=0.50, spatial_axis=1),
    RandRotate90d(keys=['image', 'label'], prob=0.20, max_k=3, spatial_axes=(0, 1)),
    RandScaleIntensityd(keys=['image'], factors=0.10, prob=0.15),
    RandShiftIntensityd(keys=['image'], offsets=0.10, prob=0.15),
    EnsureTyped(keys=['image', 'label'], dtype=torch.float32, track_meta=False),
])

infer_transforms = Compose(base_transforms + [
    EnsureTyped(keys=['image', 'label'], dtype=torch.float32, track_meta=False),
])

DatasetClassTrain = CacheDataset if CACHE_RATE_TRAIN > 0 else Dataset
DatasetClassVal = CacheDataset if CACHE_RATE_VAL_TEST > 0 else Dataset

train_ds = DatasetClassTrain(data=train_files, transform=train_transforms, cache_rate=CACHE_RATE_TRAIN, num_workers=NUM_WORKERS) if DatasetClassTrain is CacheDataset else DatasetClassTrain(data=train_files, transform=train_transforms)
val_ds = DatasetClassVal(data=val_files, transform=infer_transforms, cache_rate=CACHE_RATE_VAL_TEST, num_workers=NUM_WORKERS) if DatasetClassVal is CacheDataset else DatasetClassVal(data=val_files, transform=infer_transforms)
test_ds = DatasetClassVal(data=test_files, transform=infer_transforms, cache_rate=CACHE_RATE_VAL_TEST, num_workers=NUM_WORKERS) if DatasetClassVal is CacheDataset else DatasetClassVal(data=test_files, transform=infer_transforms)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=list_data_collate,
)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print('Train/val/test cases:', len(train_files), len(val_files), len(test_files))
first_batch = next(iter(train_loader))
print('Example train image batch:', tuple(first_batch['image'].shape))
print('Example train label batch:', tuple(first_batch['label'].shape), first_batch['label'].dtype)


Loading dataset: 100%|██████████| 9/9 [00:02<00:00,  3.64it/s]

Train/val/test cases: 99 10 11


Example train image batch: (2, 1, 96, 96, 32)
Example train label batch: (2, 1, 96, 96, 32) torch.float32


## 4. ResidualUNet3D model and training helpers

The baseline uses MONAI `SegResNet` as a robust ResidualUNet3D family model: residual blocks, encoder-decoder hierarchy, skip connections, and 3D convolutions. Hyperparameters are intentionally conservative for Kaggle T4.


In [6]:
import inspect
from torch.cuda.amp import GradScaler, autocast
from monai.networks.nets import SegResNet
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose as MonaiCompose


def create_model():
    params = inspect.signature(SegResNet).parameters
    kwargs = {
        'spatial_dims': 3,
        'init_filters': BASE_CHANNELS,
        'in_channels': 1,
        'out_channels': 2,
        'dropout_prob': DROPOUT,
        'blocks_down': (1, 2, 2, 4),
        'blocks_up': (1, 1, 1),
    }
    # Older MONAI versions may not expose all keyword names; keep the constructor adaptive.
    kwargs = {k: v for k, v in kwargs.items() if k in params}
    return SegResNet(**kwargs)


def checkpoint_payload(epoch, model, optimizer, scheduler, best_val_dice, history):
    return {
        'epoch': int(epoch),
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'best_val_dice': float(best_val_dice),
        'history': history,
        'config': {
            'model_name': MODEL_NAME,
            'architecture': 'ResidualUNet3D / MONAI SegResNet',
            'roi_size': ROI_SIZE,
            'base_channels': BASE_CHANNELS,
            'dropout': DROPOUT,
            'max_epochs': MAX_EPOCHS,
            'val_interval': VAL_INTERVAL,
            'batch_size': BATCH_SIZE,
            'num_samples_per_case': NUM_SAMPLES_PER_CASE,
            'learning_rate': LEARNING_RATE,
            'weight_decay': WEIGHT_DECAY,
            'split_sizes': {'train': len(train_files), 'val': len(val_files), 'test': len(test_files)},
            'seed': SEED,
        },
    }


def safe_torch_load(path, map_location='cpu'):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def load_checkpoint_if_available(model, optimizer=None, scheduler=None):
    latest = CHECKPOINT_DIR / 'checkpoint_latest.pt'
    if not (RESUME_TRAINING and latest.exists()):
        return 0, -1.0, []
    ckpt = safe_torch_load(latest, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    if optimizer is not None and ckpt.get('optimizer_state_dict') is not None:
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if scheduler is not None and ckpt.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = int(ckpt.get('epoch', -1)) + 1
    best_val_dice = float(ckpt.get('best_val_dice', -1.0))
    history = ckpt.get('history', [])
    print('Resumed from:', latest)
    print('Start epoch:', start_epoch, 'best_val_dice:', best_val_dice)
    return start_epoch, best_val_dice, history


post_pred = MonaiCompose([Activations(softmax=True), AsDiscrete(argmax=True, to_onehot=2)])
post_label = AsDiscrete(to_onehot=2)
dice_metric = DiceMetric(include_background=False, reduction='mean')


def foreground_dice_from_logits(logits, labels):
    pred = torch.argmax(logits, dim=1, keepdim=True) == 1
    gt = labels.long()
    if gt.ndim == 4:
        gt = gt.unsqueeze(1)
    gt = gt == 1
    reduce_dims = tuple(range(1, pred.ndim))
    intersection = (pred & gt).sum(dim=reduce_dims).float()
    denom = pred.sum(dim=reduce_dims).float() + gt.sum(dim=reduce_dims).float()
    return torch.where(denom > 0, 2.0 * intersection / denom, torch.ones_like(denom))


def run_validation(model, loader, device):
    model.eval()
    case_dices = []
    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            labels = batch['label'].to(device).long()
            with autocast(enabled=AMP and device.type == 'cuda'):
                outputs = sliding_window_inference(
                    images,
                    roi_size=ROI_SIZE,
                    sw_batch_size=SW_BATCH_SIZE,
                    predictor=model,
                    overlap=SLIDING_OVERLAP,
                    mode='gaussian',
                )
            case_dices.extend(foreground_dice_from_logits(outputs.detach(), labels).detach().cpu().tolist())
    if not case_dices:
        return 0.0
    return float(np.mean(case_dices))


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if IS_KAGGLE and device.type != 'cuda':
    raise RuntimeError('Kaggle GPU is not available. Enable T4 GPU before training.')

model = create_model().to(device)
loss_function = DiceCELoss(to_onehot_y=True, softmax=True, include_background=False)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
scaler = GradScaler(enabled=AMP and device.type == 'cuda')

num_params = sum(p.numel() for p in model.parameters())
print('Device:', device)
print('Model :', type(model).__name__)
print('Params:', f'{num_params / 1e6:.2f}M')


Device: cuda
Model : SegResNet
Params: 10.57M


/tmp/ipykernel_23/1804939320.py:124: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=AMP and device.type == 'cuda')


## 5. Train

Default is 250 epochs to match the 02_12-style training budget. If Kaggle runtime is tight, keep the same split and record any reduced epoch budget in the final comparison table.


In [7]:
# Validation hotfix for Kaggle MONAI MetaTensor issue.
# Run this cell before training/resuming if an older kernel still has the old run_validation function.
VALIDATION_HOTFIX_VERSION = 'no_decollate_tensor_dice_v2'


def foreground_dice_from_logits(logits, labels):
    pred = torch.argmax(logits, dim=1, keepdim=True) == 1
    gt = labels.long()
    if gt.ndim == 4:
        gt = gt.unsqueeze(1)
    gt = gt == 1
    if pred.shape[2:] != gt.shape[2:]:
        min_shape = tuple(min(pred.shape[d], gt.shape[d]) for d in range(2, pred.ndim))
        crop = (slice(None), slice(None), *[slice(0, s) for s in min_shape])
        pred = pred[crop]
        gt = gt[crop]
    reduce_dims = tuple(range(1, pred.ndim))
    intersection = (pred & gt).sum(dim=reduce_dims).float()
    denom = pred.sum(dim=reduce_dims).float() + gt.sum(dim=reduce_dims).float()
    return torch.where(denom > 0, 2.0 * intersection / denom, torch.ones_like(denom))


def run_validation(model, loader, device):
    model.eval()
    case_dices = []
    with torch.no_grad():
        for batch in loader:
            images = torch.as_tensor(batch['image']).to(device)
            labels = torch.as_tensor(batch['label']).to(device).long()
            with autocast(enabled=AMP and device.type == 'cuda'):
                outputs = sliding_window_inference(
                    images,
                    roi_size=ROI_SIZE,
                    sw_batch_size=SW_BATCH_SIZE,
                    predictor=model,
                    overlap=SLIDING_OVERLAP,
                    mode='gaussian',
                )
            case_dices.extend(foreground_dice_from_logits(outputs.detach(), labels).detach().cpu().tolist())
    if not case_dices:
        return 0.0
    return float(np.mean(case_dices))


print('Validation hotfix active:', VALIDATION_HOTFIX_VERSION)


Validation hotfix active: no_decollate_tensor_dice_v2


In [8]:
if RUN_TRAIN:
    start_epoch, best_val_dice, history = load_checkpoint_if_available(model, optimizer, scheduler)
    latest_ckpt = CHECKPOINT_DIR / 'checkpoint_latest.pt'
    best_ckpt = CHECKPOINT_DIR / 'checkpoint_best.pt'

    print('Training from epoch:', start_epoch)
    print('Best val Dice so far:', best_val_dice)
    t0_all = time.time()

    for epoch in range(start_epoch, MAX_EPOCHS):
        model.train()
        epoch_loss = 0.0
        step_count = 0
        t0 = time.time()

        for batch in train_loader:
            images = batch['image'].to(device)
            labels = batch['label'].to(device).long()

            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=AMP and device.type == 'cuda'):
                outputs = model(images)
                loss = loss_function(outputs, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += float(loss.detach().cpu())
            step_count += 1

        scheduler.step()
        mean_loss = epoch_loss / max(step_count, 1)
        record = {
            'epoch': epoch + 1,
            'train_loss': mean_loss,
            'lr': float(optimizer.param_groups[0]['lr']),
            'seconds': round(time.time() - t0, 2),
        }

        should_validate = ((epoch + 1) % VAL_INTERVAL == 0) or (epoch + 1 == MAX_EPOCHS)
        if should_validate:
            val_dice = run_validation(model, val_loader, device)
            record['val_dice'] = val_dice
            print(f"Epoch {epoch + 1:03d}/{MAX_EPOCHS} loss={mean_loss:.4f} val_dice={val_dice:.4f} lr={record['lr']:.2e} time={record['seconds']}s")
            if val_dice > best_val_dice:
                best_val_dice = val_dice
                torch.save(checkpoint_payload(epoch, model, optimizer, scheduler, best_val_dice, history + [record]), best_ckpt)
                print('Saved new best checkpoint:', best_ckpt)
        else:
            print(f"Epoch {epoch + 1:03d}/{MAX_EPOCHS} loss={mean_loss:.4f} lr={record['lr']:.2e} time={record['seconds']}s")

        history.append(record)
        torch.save(checkpoint_payload(epoch, model, optimizer, scheduler, best_val_dice, history), latest_ckpt)
        pd.DataFrame(history).to_csv(METRIC_DIR / '02_20_residualunet3d_training_history.csv', index=False)

    print('Training done in min:', round((time.time() - t0_all) / 60, 1))
    print('Best val Dice:', best_val_dice)
else:
    print('Skipped training. Set RUN_TRAIN=True to train.')


Training from epoch: 0
Best val Dice so far: -1.0


/tmp/ipykernel_23/1216433925.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=AMP and device.type == 'cuda'):


Epoch 001/250 loss=1.1571 lr=1.00e-04 time=34.05s
Epoch 002/250 loss=1.1045 lr=1.00e-04 time=18.69s
Epoch 003/250 loss=1.0823 lr=1.00e-04 time=19.59s
Epoch 004/250 loss=1.0665 lr=9.99e-05 time=19.8s


/tmp/ipykernel_23/3464191966.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=AMP and device.type == 'cuda'):
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:226: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  win_data = torch.cat([inputs[win_slice] for win_slice in unravel_slice]).to(sw_device)
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:370: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 th

Epoch 005/250 loss=1.0536 val_dice=0.0000 lr=9.99e-05 time=20.37s
Saved new best checkpoint: /kaggle/working/outputs_02_20_kaggle_residualunet3d_phe_only_fair_baseline/checkpoints/checkpoint_best.pt
Epoch 006/250 loss=1.0432 lr=9.99e-05 time=18.49s
Epoch 007/250 loss=1.0348 lr=9.98e-05 time=20.32s
Epoch 008/250 loss=1.0236 lr=9.97e-05 time=18.97s
Epoch 009/250 loss=1.0132 lr=9.97e-05 time=19.2s
Epoch 010/250 loss=0.9986 val_dice=0.0232 lr=9.96e-05 time=19.26s
Saved new best checkpoint: /kaggle/working/outputs_02_20_kaggle_residualunet3d_phe_only_fair_baseline/checkpoints/checkpoint_best.pt
Epoch 011/250 loss=0.9880 lr=9.95e-05 time=20.28s
Epoch 012/250 loss=0.9668 lr=9.94e-05 time=18.47s
Epoch 013/250 loss=0.9423 lr=9.93e-05 time=18.94s
Epoch 014/250 loss=0.9432 lr=9.92e-05 time=19.16s
Epoch 015/250 loss=0.9217 val_dice=0.1088 lr=9.91e-05 time=19.76s
Saved new best checkpoint: /kaggle/working/outputs_02_20_kaggle_residualunet3d_phe_only_fair_baseline/checkpoints/checkpoint_best.pt
Epoc

## 6. Predict test set and evaluate

Inference uses full-volume sliding-window prediction. Metrics are computed on the locked 11 test cases.


In [9]:
from scipy.ndimage import binary_erosion, generate_binary_structure, distance_transform_edt


def dice_score(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    denom = pred.sum() + gt.sum()
    if denom == 0:
        return 1.0
    return float(2.0 * np.logical_and(pred, gt).sum() / denom)


def binary_metrics(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    tp = int(np.logical_and(pred, gt).sum())
    fp = int(np.logical_and(pred, ~gt).sum())
    fn = int(np.logical_and(~pred, gt).sum())
    tn = int(np.logical_and(~pred, ~gt).sum())
    precision = tp / (tp + fp) if (tp + fp) else (1.0 if gt.sum() == 0 else 0.0)
    recall = tp / (tp + fn) if (tp + fn) else (1.0 if pred.sum() == 0 else 0.0)
    specificity = tn / (tn + fp) if (tn + fp) else 1.0
    return tp, fp, fn, tn, float(precision), float(recall), float(specificity)


def hd95_mm(pred, gt, spacing):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    if not pred.any() and not gt.any():
        return 0.0
    if pred.any() != gt.any():
        return float('inf')
    structure = generate_binary_structure(3, 1)
    pred_surface = np.logical_xor(pred, binary_erosion(pred, structure=structure, border_value=0))
    gt_surface = np.logical_xor(gt, binary_erosion(gt, structure=structure, border_value=0))
    if not pred_surface.any() or not gt_surface.any():
        return float('inf')
    dt_gt = distance_transform_edt(~gt_surface, sampling=spacing)
    dt_pred = distance_transform_edt(~pred_surface, sampling=spacing)
    distances = np.concatenate([dt_gt[pred_surface], dt_pred[gt_surface]])
    return float(np.percentile(distances, 95))


def volume_ml(mask, spacing):
    return float(mask.astype(bool).sum() * np.prod(spacing) / 1000.0)


def load_best_for_inference(model):
    best_ckpt = CHECKPOINT_DIR / 'checkpoint_best.pt'
    latest_ckpt = CHECKPOINT_DIR / 'checkpoint_latest.pt'
    ckpt_path = best_ckpt if best_ckpt.exists() else latest_ckpt
    if not ckpt_path.exists():
        raise FileNotFoundError(f'No checkpoint found in {CHECKPOINT_DIR}. Train first or upload checkpoint.')
    try:
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    except TypeError:
        ckpt = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    print('Loaded checkpoint:', ckpt_path)
    print('Checkpoint best_val_dice:', ckpt.get('best_val_dice'))
    return ckpt_path


if RUN_PREDICT_AND_EVALUATE:
    ckpt_path = load_best_for_inference(model)
    model.to(device)
    model.eval()
    PRED_DIR.mkdir(parents=True, exist_ok=True)

    rows = []
    with torch.no_grad():
        for batch, info in zip(test_loader, test_files):
            case_id = info['case_id']
            images = batch['image'].to(device)
            with autocast(enabled=AMP and device.type == 'cuda'):
                logits = sliding_window_inference(
                    images,
                    roi_size=ROI_SIZE,
                    sw_batch_size=SW_BATCH_SIZE,
                    predictor=model,
                    overlap=SLIDING_OVERLAP,
                    mode='gaussian',
                )
            pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

            gt_nii = nib.load(str(info['label']))
            gt = (np.asanyarray(gt_nii.dataobj) > 0).astype(np.uint8)
            image_nii = nib.load(str(info['image']))
            spacing = tuple(float(x) for x in image_nii.header.get_zooms()[:3])
            original_shape = gt.shape
            pred = pred[:original_shape[0], :original_shape[1], :original_shape[2]]

            pred_path = PRED_DIR / f'{case_id}.nii.gz'
            pred_img = nib.Nifti1Image(pred.astype(np.uint8), affine=gt_nii.affine, header=gt_nii.header)
            pred_img.set_data_dtype(np.uint8)
            nib.save(pred_img, str(pred_path))

            tp, fp, fn, tn, precision, recall, specificity = binary_metrics(pred, gt)
            rows.append({
                'case_id': case_id,
                'checkpoint': str(ckpt_path),
                'prediction_path': str(pred_path),
                'dice': dice_score(pred, gt),
                'precision': precision,
                'sensitivity': recall,
                'recall': recall,
                'specificity': specificity,
                'hd95_mm': hd95_mm(pred, gt, spacing),
                'gt_volume_ml': volume_ml(gt, spacing),
                'pred_volume_ml': volume_ml(pred, spacing),
                'tp': tp,
                'fp': fp,
                'fn': fn,
                'tn': tn,
                'spacing': spacing,
                'shape': original_shape,
            })
            print(f"{case_id}: Dice={rows[-1]['dice']:.4f}, sens={recall:.4f}, pred_ml={rows[-1]['pred_volume_ml']:.2f}, gt_ml={rows[-1]['gt_volume_ml']:.2f}")

    metrics_df = pd.DataFrame(rows)
    metrics_csv = METRIC_DIR / '02_20_residualunet3d_test_metrics.csv'
    metrics_df.to_csv(metrics_csv, index=False)

    finite_hd95 = metrics_df['hd95_mm'].replace([np.inf, -np.inf], np.nan)
    summary = {
        'model': MODEL_NAME,
        'checkpoint': str(ckpt_path),
        'n_test': int(len(metrics_df)),
        'dice_mean': float(metrics_df['dice'].mean()),
        'dice_median': float(metrics_df['dice'].median()),
        'sensitivity_mean': float(metrics_df['sensitivity'].mean()),
        'precision_mean': float(metrics_df['precision'].mean()),
        'specificity_mean': float(metrics_df['specificity'].mean()),
        'hd95_mm_mean_finite': float(finite_hd95.mean()) if finite_hd95.notna().any() else None,
        'gt_volume_ml_mean': float(metrics_df['gt_volume_ml'].mean()),
        'pred_volume_ml_mean': float(metrics_df['pred_volume_ml'].mean()),
        'fairness': {
            'dataset': 'PHE-SICH CT only',
            'target': 'manual PHE binary mask',
            'train_cases': len(train_files),
            'val_cases': len(val_files),
            'test_cases': len(test_files),
            'no_ich_teacher': True,
            'no_pseudo_ich': True,
            'no_prior_channel': True,
        },
    }
    summary_json = METRIC_DIR / '02_20_residualunet3d_test_summary.json'
    summary_json.write_text(json.dumps(summary, indent=2), encoding='utf-8')

    print('Metrics CSV :', metrics_csv)
    print('Summary JSON:', summary_json)
    print(json.dumps(summary, indent=2))
    display(metrics_df)
else:
    print('Skipped prediction/evaluation. Set RUN_PREDICT_AND_EVALUATE=True after training.')


Loaded checkpoint: /kaggle/working/outputs_02_20_kaggle_residualunet3d_phe_only_fair_baseline/checkpoints/checkpoint_best.pt
Checkpoint best_val_dice: 0.2766076818108559


/tmp/ipykernel_23/1654477056.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=AMP and device.type == 'cuda'):


0002: Dice=0.3019, sens=0.2691, pred_ml=1.53, gt_ml=1.95
0029: Dice=0.3465, sens=0.5351, pred_ml=8.14, gt_ml=3.90
0033: Dice=0.3850, sens=0.2814, pred_ml=3.05, gt_ml=6.61
0051: Dice=0.0103, sens=0.0422, pred_ml=2.64, gt_ml=0.37
0061: Dice=0.0059, sens=0.0197, pred_ml=2.74, gt_ml=0.49
0084: Dice=0.3474, sens=0.2772, pred_ml=1.64, gt_ml=2.75
0095: Dice=0.3738, sens=0.3470, pred_ml=4.85, gt_ml=5.66
0096: Dice=0.3178, sens=0.2844, pred_ml=6.04, gt_ml=7.65
0113: Dice=0.0486, sens=0.1417, pred_ml=1.38, gt_ml=0.29
0116: Dice=0.2394, sens=0.6974, pred_ml=3.80, gt_ml=0.79
0167: Dice=0.0000, sens=0.0000, pred_ml=1.46, gt_ml=1.09
Metrics CSV : /kaggle/working/outputs_02_20_kaggle_residualunet3d_phe_only_fair_baseline/metrics/02_20_residualunet3d_test_metrics.csv
Summary JSON: /kaggle/working/outputs_02_20_kaggle_residualunet3d_phe_only_fair_baseline/metrics/02_20_residualunet3d_test_summary.json
{
  "model": "monai_segresnet_residualunet3d_phe_only",
  "checkpoint": "/kaggle/working/outputs_02_20

,case_id,checkpoint,prediction_path,dice,precision,sensitivity,recall,specificity,hd95_mm,gt_volume_ml,pred_volume_ml,tp,fp,fn,tn,spacing,shape
0,0002,/kaggle/working/outputs_02_20_kaggle_residualu...,/kaggle/working/outputs_02_20_kaggle_residualu...,0.301887,0.343750,0.269113,0.269113,0.999900,55.932783,1.949072,1.525879,440,840,1195,8386133,"(0.48828125, 0.48828125, 5.0)","(512, 512, 32)"
1,0029,/kaggle/working/outputs_02_20_kaggle_residualu...,/kaggle/working/outputs_02_20_kaggle_residualu...,0.346472,0.256181,0.535054,0.535054,0.999218,21.096428,3.897568,8.140396,2259,6559,1963,8377827,"(0.4296875, 0.4296875, 5.0)","(512, 512, 32)"
2,0033,/kaggle/working/outputs_02_20_kaggle_residualu...,/kaggle/working/outputs_02_20_kaggle_residualu...,0.384995,0.609137,0.281436,0.281436,0.999881,11.677552,6.607771,3.052950,1560,1001,3983,8382064,"(0.48828125, 0.48828125, 5.0)","(512, 512, 32)"
3,0051,/kaggle/working/outputs_02_20_kaggle_residualu...,/kaggle/working/outputs_02_20_kaggle_residualu...,0.010317,0.005877,0.042208,0.042208,0.999664,47.982051,0.367165,2.636909,13,2199,295,6551093,"(0.48828125, 0.48828125, 5.0)","(512, 512, 25)"
4,0061,/kaggle/working/outputs_02_20_kaggle_residualu...,/kaggle/working/outputs_02_20_kaggle_residualu...,0.005911,0.003478,0.019656,0.019656,0.999650,64.029959,0.485182,2.741814,8,2292,399,6550901,"(0.48828125, 0.48828125, 5.0)","(512, 512, 25)"
5,0084,/kaggle/working/outputs_02_20_kaggle_residualu...,/kaggle/working/outputs_02_20_kaggle_residualu...,0.347426,0.465231,0.277228,0.277228,0.999896,49.026620,2.751509,1.639605,756,869,1971,8385012,"(0.44921875, 0.44921875, 5.0)","(512, 512, 32)"
6,0095,/kaggle/working/outputs_02_20_kaggle_residualu...,/kaggle/working/outputs_02_20_kaggle_residualu...,0.373757,0.404953,0.347023,0.347023,0.999779,12.690302,5.660169,4.850459,1259,1850,2369,8383130,"(0.55859375, 0.55859375, 5.0)","(512, 512, 32)"
7,0096,/kaggle/working/outputs_02_20_kaggle_residualu...,/kaggle/working/outputs_02_20_kaggle_residualu...,0.317834,0.360103,0.284445,0.284445,0.999587,7.402859,7.648468,6.041527,1825,3243,4591,7854661,"(0.48828125, 0.48828125, 5.0)","(512, 512, 30)"
8,0113,/kaggle/working/outputs_02_20_kaggle_residualu...,/kaggle/working/outputs_02_20_kaggle_residualu...,0.048641,0.029361,0.141667,0.141667,0.999828,45.090080,0.286102,1.380444,34,1124,206,6552236,"(0.48828125, 0.48828125, 5.0)","(512, 512, 25)"
9,0116,/kaggle/working/outputs_02_20_kaggle_residualu...,/kaggle/working/outputs_02_20_kaggle_residualu...,0.239418,0.144514,0.697428,0.697428,0.999675,38.239802,0.787973,3.802776,461,2729,200,8385218,"(0.48828125, 0.48828125, 5.0)","(512, 512, 32)"


## 7. Experiment registry

This small registry row makes it easier to compare against 02_12, 16b, 16c, 18, 13+14, and 13b+14b in the report.


In [10]:
registry = {
    'experiment': '02_20',
    'notebook': '02_20_kaggle_residualunet3d_phe_only_fair_baseline.ipynb',
    'model': MODEL_NAME,
    'architecture': 'ResidualUNet3D / MONAI SegResNet',
    'framework': 'PyTorch + MONAI',
    'run_target': 'Kaggle T4',
    'dataset': 'PHE-SICH CT-IDS SubdatasetA_NIFIT/NIFIT',
    'input_channels': 'CT only',
    'label': 'manual PHE binary mask',
    'train_cases': len(train_files),
    'val_cases': len(val_files),
    'test_cases': len(test_files),
    'split_source': 'locked 02_12 split ids',
    'max_epochs': MAX_EPOCHS,
    'roi_size': str(ROI_SIZE),
    'base_channels': BASE_CHANNELS,
    'dropout': DROPOUT,
    'amp': AMP,
    'no_ich_teacher': True,
    'no_pseudo_ich': True,
    'no_prior_channel': True,
    'output_root': str(OUTPUT_ROOT),
}

summary_path = METRIC_DIR / '02_20_residualunet3d_test_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    registry.update({
        'test_dice_mean': summary.get('dice_mean'),
        'test_dice_median': summary.get('dice_median'),
        'test_sensitivity_mean': summary.get('sensitivity_mean'),
        'test_precision_mean': summary.get('precision_mean'),
        'test_hd95_mm_mean_finite': summary.get('hd95_mm_mean_finite'),
    })

registry_csv = OUTPUT_ROOT / '02_20_residualunet3d_experiment_registry.csv'
pd.DataFrame([registry]).to_csv(registry_csv, index=False)
print('Registry:', registry_csv)
display(pd.DataFrame([registry]))


Registry: /kaggle/working/outputs_02_20_kaggle_residualunet3d_phe_only_fair_baseline/02_20_residualunet3d_experiment_registry.csv


,experiment,notebook,model,architecture,framework,run_target,dataset,input_channels,label,train_cases,...,amp,no_ich_teacher,no_pseudo_ich,no_prior_channel,output_root,test_dice_mean,test_dice_median,test_sensitivity_mean,test_precision_mean,test_hd95_mm_mean_finite
0,02_20,02_20_kaggle_residualunet3d_phe_only_fair_base...,monai_segresnet_residualunet3d_phe_only,ResidualUNet3D / MONAI SegResNet,PyTorch + MONAI,Kaggle T4,PHE-SICH CT-IDS SubdatasetA_NIFIT/NIFIT,CT only,manual PHE binary mask,99,...,True,True,True,True,/kaggle/working/outputs_02_20_kaggle_residualu...,0.21606,0.301887,0.263205,0.238417,34.877077


## Notes for scientific comparison

`02_20` is an architecture-control experiment. It should be compared to `02_12`, `17`, `18b`, and similar PHE-only baselines under the same split and test metrics.

Do not interpret it as a prior/teacher experiment: it intentionally does not use ICH labels, pseudo ICH, or spatial priors. If runtime requires fewer than 250 epochs, report the actual epoch budget from `02_20_residualunet3d_training_history.csv`.
